# CPDS-AI: YOLOv8 Adult vs Child Classification

Notebook này được thiết kế để chạy trên Kaggle/Google Colab. Mục tiêu là fine-tune mô hình **YOLOv8n** trên tập dữ liệu Adult/Child, sau đó xuất ra file ONNX để chạy trên máy tính cá nhân và mạch Orange Pi 5.

In [ ]:
!pip install -q ultralytics onnx
import ultralytics
ultralytics.checks()

## 1. Chọn Kaggle Dataset

Upload/export dataset YOLOv8 (có `data.yaml`) thành một Kaggle Dataset rồi gắn nó vào notebook. Không đưa Roboflow API key vào notebook hoặc GitHub.

In [ ]:
from pathlib import Path

DATASET_YAML = Path("/kaggle/input/cpds-adult-child/data.yaml")  # sửa tên mount nếu cần
if not DATASET_YAML.is_file():
    raise FileNotFoundError(f"Attach the YOLO dataset and set DATASET_YAML; missing {DATASET_YAML}")
print(DATASET_YAML.read_text())

## 2. Huấn luyện (Train) YOLOv8n

In [ ]:
from ultralytics import YOLO

# Load pre-trained model nano
model = YOLO('yolov8n.pt')

# Kaggle GPU: device=0. CPU: đổi thành device='cpu'.
results = model.train(data=str(DATASET_YAML), epochs=20, imgsz=640, device=0, project="runs", name="adult_child", exist_ok=True)

## 3. Chuyển đổi định dạng sang ONNX (Export)

In [ ]:
# Load best weights
best_model = YOLO('runs/adult_child/weights/best.pt')

# Export sang ONNX format
export_path = best_model.export(format='onnx', opset=12, dynamic=True, simplify=True)
print(f"Model exported to: {export_path}")